<a href="https://colab.research.google.com/github/sambireddyakkala/Student_performance_prediction/blob/main/Student_Performance_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective**: Predict a student's final grade (G3) based on their second-period grade (G2) using Machine Learning.
Project features
Upload the student dataset (student_data.csv).
Train a Linear Regression model.
Compare Gradient Descent, SGD, Ridge, Lasso and other regression algorithms.
Display MAE, MSE, RMSE and R² scores.
Enter a student's G2 marks and predict their final grade.
Display graphs and model comparisons on a dashboard.

## 1. Import libraries

Run each cell from top to bottom. Google Colab normally has the required libraries pre-installed.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 2. Load the dataset

In Colab, run the cell and select your uploaded `student_data.csv`. If the file is already present in the runtime, it will be loaded automatically.

In [ ]:
from pathlib import Path

candidate_paths = [
    "/content/student_data.csv",
    "/mnt/data/student_data.csv",
    "student_data.csv"
]
csv_path = next((p for p in candidate_paths if Path(p).exists()), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next((name for name in uploaded.keys() if name.lower().endswith(".csv")), None)
    except ImportError:
        raise FileNotFoundError("Place student_data.csv in the working folder or run this notebook in Google Colab.")

if csv_path is None:
    raise FileNotFoundError("No CSV file was selected.")

df = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
display(df.head())

## 3. Inspect and clean the data

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
display(df.dtypes.rename("data_type").to_frame())

print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False).rename("missing_count").to_frame())

print("\nDuplicate rows:", df.duplicated().sum())

required = {"G2", "G3"}
if not required.issubset(df.columns):
    raise ValueError("The dataset must contain G2 and G3 columns.")

# Keep an unchanged copy for reference, and use a working copy for modeling.
data = df.copy()

## 4. Exploratory data analysis (EDA)

The first plots show the distribution of final grades, the relationship between `G2` and `G3`, and the correlation of numerical features with `G3`.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(data["G3"].dropna(), bins=range(int(data["G3"].min()), int(data["G3"].max()) + 2),
         edgecolor="black")
plt.title("Distribution of Final Grades (G3)")
plt.xlabel("Final grade (G3)")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(data["G2"], data["G3"], alpha=0.65)
plt.title("Second-Period Grade (G2) vs Final Grade (G3)")
plt.xlabel("G2")
plt.ylabel("G3")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

numeric_data = data.select_dtypes(include=np.number)
correlations = numeric_data.corr(numeric_only=True)["G3"].drop("G3").sort_values()
plt.figure(figsize=(7, 4))
correlations.plot(kind="barh")
plt.title("Correlation of Numerical Features with G3")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

display(correlations.sort_values(ascending=False).to_frame("correlation_with_G3"))
print(f"Correlation between G2 and G3: {data['G2'].corr(data['G3']):.4f}")

## 5. Prepare a reproducible train/test split

- **Training set (80%)**: used to fit model parameters.
- **Test set (20%)**: held aside to evaluate predictions on unseen rows.
- A fixed random seed makes the split repeatable.

For a fair comparison, the models below use the same train/test row split. The simple models use `G2`; the multiple-feature models use `G1`, `G2`, `studytime`, `failures`, and `absences`, matching the uploaded report's feature plan.

In [ ]:
train_idx, test_idx = train_test_split(
    data.index, test_size=0.20, random_state=RANDOM_STATE
)
y_train = data.loc[train_idx, "G3"].astype(float)
y_test = data.loc[test_idx, "G3"].astype(float)

Xg2_train = data.loc[train_idx, ["G2"]].astype(float)
Xg2_test = data.loc[test_idx, ["G2"]].astype(float)

multiple_features = ["G1", "G2", "studytime", "failures", "absences"]
Xm_train = data.loc[train_idx, multiple_features].astype(float)
Xm_test = data.loc[test_idx, multiple_features].astype(float)

print(f"Training rows: {len(train_idx)}")
print(f"Testing rows:  {len(test_idx)}")

## 6. Define evaluation metrics

- **MAE**: average absolute prediction error; lower is better.
- **MSE**: average squared error; lower is better.
- **RMSE**: square root of MSE, in grade units; lower is better.
- **R²**: proportion of target variance explained by the model; usually higher is better, and it can be negative on poor predictions.

In [ ]:
results = []
predictions = {}

def evaluate_model(name, y_true, y_pred, feature_group):
    y_pred = np.asarray(y_pred).reshape(-1)
    mse = mean_squared_error(y_true, y_pred)
    row = {
        "Model": name,
        "Features": feature_group,
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }
    results.append(row)
    predictions[name] = y_pred
    return row

## 7. Model A — Simple Linear Regression

This model uses only `G2` to estimate `G3`. It is easy to explain and useful as a baseline.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(Xg2_train, y_train)
pred_linear = linear_model.predict(Xg2_test)
evaluate_model("Simple Linear Regression", y_test, pred_linear, "G2")

print(f"Regression equation: G3 = {linear_model.coef_[0]:.4f} × G2 + {linear_model.intercept_:.4f}")

## 8. Models B–D — Batch, Stochastic and Mini-batch Gradient Descent

Fast version: uses 250 epochs for each scratch gradient-descent model to reduce runtime. For this small dataset, this should run quickly in Colab; scores can vary slightly from a longer training run.

These implementations use standardized `G2` values so gradient updates are on a stable scale. Batch Gradient Descent uses every training row per update; Stochastic Gradient Descent uses one row; Mini-batch Gradient Descent uses a small group of rows.


In [ ]:
class ScratchGradientDescent:
    def __init__(self, learning_rate=0.03, epochs=250, method="batch",
                 batch_size=32, random_state=42):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.method = method
        self.batch_size = batch_size
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(X)
        Xb = np.c_[np.ones(len(Xs)), Xs]
        rng = np.random.default_rng(self.random_state)
        self.theta_ = np.zeros(Xb.shape[1], dtype=float)
        n = len(y)

        for _ in range(self.epochs):
            if self.method == "batch":
                grad = (2 / n) * Xb.T @ (Xb @ self.theta_ - y)
                self.theta_ -= self.learning_rate * grad
            else:
                order = rng.permutation(n)
                if self.method == "stochastic":
                    batches = [order[i:i+1] for i in range(n)]
                else:
                    batches = [order[i:i+self.batch_size] for i in range(0, n, self.batch_size)]
                for idx in batches:
                    xb_batch, y_batch = Xb[idx], y[idx]
                    grad = (2 / len(idx)) * xb_batch.T @ (xb_batch @ self.theta_ - y_batch)
                    self.theta_ -= self.learning_rate * grad
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        Xs = self.scaler_.transform(X)
        Xb = np.c_[np.ones(len(Xs)), Xs]
        return Xb @ self.theta_

batch_gd = ScratchGradientDescent(method="batch", learning_rate=0.03, epochs=250)
batch_gd.fit(Xg2_train, y_train)
evaluate_model("Batch Gradient Descent", y_test, batch_gd.predict(Xg2_test), "G2")

sgd_scratch = ScratchGradientDescent(method="stochastic", learning_rate=0.0005, epochs=80)
sgd_scratch.fit(Xg2_train, y_train)
evaluate_model("Stochastic Gradient Descent", y_test, sgd_scratch.predict(Xg2_test), "G2")

mini_gd = ScratchGradientDescent(method="mini", learning_rate=0.03, epochs=400, batch_size=32)
mini_gd.fit(Xg2_train, y_train)
evaluate_model("Mini-batch Gradient Descent", y_test, mini_gd.predict(Xg2_test), "G2")

print("Gradient Descent models trained.")

## 9. Model E — Scikit-learn SGD Regressor

This is a library implementation of stochastic optimization. Its result may differ from the scratch implementation because its update rules and convergence settings differ.

In [ ]:
sgd_library = Pipeline([
    ("scale", StandardScaler()),
    ("model", SGDRegressor(
        loss="squared_error", penalty=None, max_iter=1500,
        tol=1e-5, learning_rate="invscaling", eta0=0.01,
        random_state=RANDOM_STATE
    ))
])
sgd_library.fit(Xg2_train, y_train)
evaluate_model("SGDRegressor (sklearn)", y_test, sgd_library.predict(Xg2_test), "G2")

## 10. Model F — Multiple Linear Regression

This model uses five features: `G1`, `G2`, `studytime`, `failures`, and `absences`. Multiple features may capture more information than `G2` alone.

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(Xm_train, y_train)
evaluate_model("Multiple Linear Regression", y_test, multiple_model.predict(Xm_test),
               ", ".join(multiple_features))

display(pd.DataFrame({
    "Feature": multiple_features,
    "Coefficient": multiple_model.coef_
}))
print(f"Intercept: {multiple_model.intercept_:.4f}")

## 11. Model G — Polynomial Regression

Polynomial Regression adds a squared `G2` term, allowing a curved relationship while still using one input feature. Degree 2 is chosen to keep the model understandable.

In [ ]:
polynomial_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear", LinearRegression())
])
polynomial_model.fit(Xg2_train, y_train)
evaluate_model("Polynomial Regression (degree 2)", y_test,
               polynomial_model.predict(Xg2_test), "G2 and G2²")

## 12. Models H–I — Ridge and Lasso Regression

Ridge and Lasso add regularization to multiple linear regression. Ridge shrinks coefficients; Lasso can shrink some coefficients to zero. Features are standardized before regularization.

In [ ]:
ridge_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
ridge_model.fit(Xm_train, y_train)
evaluate_model("Ridge Regression", y_test, ridge_model.predict(Xm_test),
               ", ".join(multiple_features))

lasso_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", Lasso(alpha=0.05, max_iter=5000))
])
lasso_model.fit(Xm_train, y_train)
evaluate_model("Lasso Regression", y_test, lasso_model.predict(Xm_test),
               ", ".join(multiple_features))

## 13. Compare all models

The table is sorted by RMSE (lower is better). Use R² as a complementary measure. The best score on this one test split does not guarantee that a model will perform best on every dataset.

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.round(4))

best_model_name = results_df.iloc[0]["Model"]
print("Lowest test RMSE on this split:", best_model_name)

## 14. Visualize model performance

In [ ]:
plt.figure(figsize=(8, 4))
plot_df = results_df.sort_values("RMSE", ascending=True)
plt.barh(plot_df["Model"], plot_df["RMSE"])
plt.xlabel("RMSE (lower is better)")
plt.title("Regression Model Comparison — Test RMSE")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plot_df_r2 = results_df.sort_values("R2", ascending=True)
plt.barh(plot_df_r2["Model"], plot_df_r2["R2"])
plt.xlabel("R² score (higher is generally better)")
plt.title("Regression Model Comparison — Test R²")
plt.tight_layout()
plt.show()

## 15. Actual vs. predicted grades

The closer points are to the diagonal line, the closer predictions are to the actual grades. The graph below uses the model with the lowest RMSE in the comparison table.

In [ ]:
prediction_lookup = {
    "Simple Linear Regression": pred_linear,
    "Batch Gradient Descent": batch_gd.predict(Xg2_test),
    "Stochastic Gradient Descent": sgd_scratch.predict(Xg2_test),
    "Mini-batch Gradient Descent": mini_gd.predict(Xg2_test),
    "SGDRegressor (sklearn)": sgd_library.predict(Xg2_test),
    "Multiple Linear Regression": multiple_model.predict(Xm_test),
    "Polynomial Regression (degree 2)": polynomial_model.predict(Xg2_test),
    "Ridge Regression": ridge_model.predict(Xm_test),
    "Lasso Regression": lasso_model.predict(Xm_test)
}

best_pred = prediction_lookup[best_model_name]
low = min(float(y_test.min()), float(np.min(best_pred)))
high = max(float(y_test.max()), float(np.max(best_pred)))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_pred, alpha=0.7)
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "Actual_G3": y_test.to_numpy(),
    "Predicted_G3": np.round(best_pred, 2),
    "Absolute_Error": np.round(np.abs(y_test.to_numpy() - best_pred), 2)
})
display(comparison.head(10))

## 16. Predict a grade from a new G2 value

For a simple demonstration, the prediction tool below uses the Simple Linear Regression model, because it requires only one input (`G2`). Enter a value in the dataset's grade range (typically 0–20).

This is a demonstration only. It should not be treated as a reliable forecast of an individual student's future.

In [ ]:
def predict_final_grade_from_g2(g2_marks):
    if not 0 <= float(g2_marks) <= 20:
        raise ValueError("Enter G2 between 0 and 20.")
    prediction = float(linear_model.predict(pd.DataFrame({"G2": [float(g2_marks)]}))[0])
    # Keep the display within the dataset's usual grade scale.
    return round(float(np.clip(prediction, 0, 20)), 2)

# Change this example value and re-run the cell.
sample_g2 = 15
print("Input G2:", sample_g2)
print("Predicted G3:", predict_final_grade_from_g2(sample_g2))

## 17. Save the model and results

This exports the simple one-input model for future reuse and saves the model comparison table as a CSV file. In Colab, the download cell lets you save the files to your computer.

In [ ]:
model_path = "student_grade_linear_model.joblib"
results_path = "regression_model_comparison.csv"

joblib.dump(linear_model, model_path)
results_df.to_csv(results_path, index=False)

print("Saved:", model_path)
print("Saved:", results_path)

try:
    from google.colab import files
    files.download(model_path)
    files.download(results_path)
except ImportError:
    print("Files are saved in the current working directory.")